## Import

In [4]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import json
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm

## Env

In [6]:
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENROUTER_API_KEY")

## Parameters

In [7]:
RUNS = 3

MODELS = [
    "anthropic/claude-sonnet-4.5",
    "google/gemini-flash-1.5",
    "meta-llama/llama-3.3-70b-instruct",
    "qwen/qwen3.6-plus",
    "mistralai/mistral-large-2512",
]


TEMPERATURE = 0

SYSTEM_PROMPT = """
Person A and Person B are talking to each other. Person A asks a question and Person B answers.
Given Person A's question, estimate how strongly Person B's answer should be interpreted as
answering “yes” versus “no.”

Output a single integer score from 1 to 7, where:

1 = definitely no
2 = very likely no
3 = somewhat likely no
4 = completely uncertain, ambiguous
5 = somewhat likely yes
6 = very likely yes
7 = definitely yes

Also report the rationale for your decision in a few sentences, explaining why you gave the score you did.
If the answer is ambiguous or irrelevant, explain why.
"""

SCORE_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "string"},
                    "score": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 7
                    },
                    "rationale": {"type": "string"}
                },
                "required": ["id", "score", "rationale"],
                "additionalProperties": False
            }
        }
    },
    "required": ["results"],
    "additionalProperties": False
}


##  Coe

In [11]:


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

def read_qas(file_path: str, retain=None) -> list:
    """
    Read the data from the given CSV file and transform it into a json structure that can be used as batch input to the LLMs.

    Args:
        file_path (str): The path to the CSV file.
        retain (int): The number of rows to include in the output for direct and indirect qaüpairs respectively.
    """
    # read the csv file
    df = pd.read_csv(file_path, sep=";")

    # clip if necessary
    if retain is not None:
        if retain > len(df):
            raise ValueError(f"Retain value {retain} is greater than the number of qas in the dataframe {len(df)}.")
        df = df.head(retain)


    # extract only relevant information
    df_new = df[["CODE", "CONTEXT QUESTION", "CRITICAL UTTERANCE"]]
    df_new["text"]  = "Person A: " + df_new["CONTEXT QUESTION"] + "; Person B: " + df_new["CRITICAL UTTERANCE"]
    df_new.rename(columns={"CODE": "id"}, inplace=True)

    # convert into json
    output = [
    {
        "id": str(row["id"]),
        "text": row["text"]
    }
    for _, row in df_new.iterrows()
    ]

    return output



def score_qas(model: str, qas: list) -> dict:
    """
    Score the QAs using one OpenRouter model.
    """

    response = client.chat.completions.create(
        model=model,
        temperature=TEMPERATURE,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": json.dumps(qas),
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "batch_classification_scores",
                "strict": True,
                "schema": SCORE_SCHEMA,
            },
        },
        extra_body={
            "provider": {
                "require_parameters": True
            }
        },
    )

    content = response.choices[0].message.content
    return json.loads(content)



def save_result(model: str, run: int, result):
    '''Save the scoring results to a CSV file.'''
    
    # Convert results to a dataframe
    results_df = pd.DataFrame(result["results"])

    # Save to CSV
    results_df.to_csv(f"data/raw/{model}_score_run{run+1}.csv", index=False)


In [10]:
qas = read_qas("data/external/iCO-Eval2_summarizedRatings.csv", retain=5)

print(len(qas))



5


In [12]:
for model in MODELS:
    
    for run in tqdm(range(RUNS), desc=f"Currently collecting data from {model}"):
        result = score_qas(model, qas)
        save_result(model, run, result)




Currently collecting data from anthropic/claude-sonnet-4.5:   0%|          | 0/3 [00:02<?, ?it/s]


BadRequestError: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{"type":"error","error":{"type":"invalid_request_error","message":"output_config.format.schema: For \'integer\' type, properties maximum, minimum are not supported"},"request_id":"req_011CcxmXQSmwDUmBeBXFqUpV"}', 'provider_name': 'Azure', 'is_byok': False, 'previous_errors': [{'code': 400, 'message': 'Provider returned error', 'provider_name': 'Amazon Bedrock', 'raw': '{"message":"output_config.format.schema: For \'integer\' type, properties maximum, minimum are not supported"}'}, {'code': 400, 'message': 'Provider returned error', 'provider_name': 'Anthropic', 'raw': '{"type":"error","error":{"type":"invalid_request_error","message":"output_config.format.schema: For \'integer\' type, properties maximum, minimum are not supported"},"request_id":"req_011CcxmXHiTZj4ZGbTB4z8bz"}'}, {'code': 400, 'message': 'Provider returned error', 'provider_name': 'Anthropic', 'raw': '{"